In [1]:
# In [1]:
# TF1-style code from the course:
#     from keras.backend.tensorflow_backend import set_session
#     config = tf.ConfigProto(); config.gpu_options.allow_growth = True
#     config.log_device_placement = True
#     sess = tf.Session(config=config); set_session(sess)
# ConfigProto / Session / set_session were all removed in TF 2.x — eager
# execution is the default and there is no session to configure.
import tensorflow as tf

# Grow GPU memory on demand instead of grabbing it all up front
# (the TF2 equivalent of config.gpu_options.allow_growth = True).
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print("memory growth:", e)

# NOTE: the course also sets log_device_placement=True. Its TF2 equivalent,
# tf.debugging.set_log_device_placement(True), prints a line for *every* op —
# it buries the Keras progress bar and slows training down enormously.
# The one-off check below tells us what we need instead.
print("TensorFlow:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())

with tf.device("/GPU:0"):
    _probe = tf.matmul(tf.random.normal((1000, 1000)), tf.random.normal((1000, 1000)))
print("matmul ran on:", _probe.device)

TensorFlow: 2.19.1
Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
matmul ran on: /job:localhost/replica:0/task:0/device:GPU:0


2026-08-31 07:23:00.104058: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-08-31 07:23:00.104192: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-08-31 07:23:00.104195: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
I0000 00:00:1788141180.104475 1911498 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1788141180.104663 1911498 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [2]:
# In [2]:
import keras
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense, Input
from keras import regularizers
from keras.optimizers import SGD
from keras import utils
import numpy as np

In [3]:
# In [3]:
batch_size = 100
n_inputs = 784
n_classes = 10
n_epochs = 50

In [4]:
# In [4]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

In [5]:
# In [ ]:
# Reshape the two dimensional 28 x 28 pixels
# Flattened vector of 784 values
X_train = X_train.reshape(60000, n_inputs)
X_test = X_test.reshape(10000, n_inputs)

In [6]:
# In [6]:
# Convert in float 32
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

In [7]:
# In [7]:
# Normalize the values of image vectors to fit under 1
X_train /= 255
X_test /= 255

In [8]:
# In [8]:
# Convert output data into one hot encoded format
y_train = utils.to_categorical(y_train, n_classes)
y_test = utils.to_categorical(y_test, n_classes)

In [9]:
# In [9]:
y_test[1]

array([0., 0., 1., 0., 0., 0., 0., 0., 0., 0.])

In [10]:
# In [10]:
model = Sequential()
# Keras 3: declare the input shape with an Input layer
# (passing input_shape= to the first Dense is deprecated)
model.add(Input(shape=(n_inputs,)))

In [11]:
# In [11]:
# first layer = input vector
model.add(Dense(units=128, activation='sigmoid', kernel_regularizer=regularizers.l2(0.01)))

In [12]:
# In [12]:
model.add(Dense(units=128, activation='relu', kernel_regularizer=regularizers.l2(0.01)))

In [13]:
# In [ ]:
model.add(Dense(units=128, activation='relu', kernel_regularizer=regularizers.l2(0.01)))

In [14]:
# In [14]:
# output layer = number of classes
model.add(Dense(units=n_classes, activation='softmax'))

In [15]:
# In [15]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 134,794 (526.54 KB)

 Trainable params: 134,794 (526.54 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# In [16]:
model.compile(loss='categorical_crossentropy',
              optimizer=SGD(),
              metrics=['accuracy'])

In [17]:
# In [*]:  (still running)
model.fit(X_train, y_train, batch_size=batch_size, epochs=n_epochs)

Epoch 1/50


2026-08-31 07:23:00.727662: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.3973 - loss: 6.3611
Epoch 2/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.6666 - loss: 4.8403
Epoch 3/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.7646 - loss: 3.7388
Epoch 4/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8176 - loss: 3.0535
Epoch 5/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8408 - loss: 2.5659
Epoch 6/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8527 - loss: 2.1967
Epoch 7/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8603 - loss: 1.9098
Epoch 8/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8662 - loss: 1.6846
Epoch 9/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8711 - loss: 1.5070
Epoch 10/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8738 - loss: 1.3657
Epoch 11/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8764 - loss: 1.2538
Epoch 12/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy

In [18]:
# In [18]:
scores = model.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8928 - loss: 0.7048


In [19]:
# In [19]:
print('loss:\n', scores[0])
print('accuracy:\n', scores[1])

loss:
 0.7047580480575562
accuracy:
 0.892799973487854


In [20]:
# In [20]:
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [21]:
# In [*]:  (still running — recompiled with adam, retraining)
model.fit(X_train, y_train, batch_size=batch_size, epochs=n_epochs)

Epoch 1/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8752 - loss: 0.7534
Epoch 2/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8769 - loss: 0.7308
Epoch 3/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8745 - loss: 0.7222
Epoch 4/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8773 - loss: 0.7038
Epoch 5/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8770 - loss: 0.6949
Epoch 6/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8786 - loss: 0.6814
Epoch 7/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8780 - loss: 0.6761
Epoch 8/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8804 - loss: 0.6659
Epoch 9/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8802 - loss: 0.6586
Epoch 10/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8839 - loss: 0.6480
Epoch 11/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8805 - loss: 0.6506
Epoch 12/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 

In [22]:
scores = model.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9184 - loss: 0.4871


In [23]:
print('loss:\n', scores[0])
print('accuracy:\n', scores[1])

loss:
 0.487059623003006
accuracy:
 0.91839998960495
